In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW:", DATA_RAW)

PROJECT_ROOT: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand
DATA_RAW: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand\data\raw


In [3]:
train = pd.read_csv(DATA_RAW / "train.csv")

print("train shape:", train.shape)
display(train.head())

train shape: (10886, 12)


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


In [4]:
train["datetime"] = pd.to_datetime(train["datetime"])

train["year"] = train["datetime"].dt.year
train["month"] = train["datetime"].dt.month
train["day"] = train["datetime"].dt.day
train["hour"] = train["datetime"].dt.hour
train["weekday"] = train["datetime"].dt.weekday
train["is_weekend"] = train["weekday"].isin([5, 6]).astype(int)

display(train.head())

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,year,month,day,hour,weekday,is_weekend
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16,2011,1,1,0,5,1
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40,2011,1,1,1,5,1
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32,2011,1,1,2,5,1
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13,2011,1,1,3,5,1
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1,2011,1,1,4,5,1


In [5]:
target_col = "count"
leakage_cols = ["casual", "registered"]

excluded_cols = [
    target_col,
    "casual",
    "registered",
    "datetime",
    "day",  # excluded from main comparison because of transfer risk
]

feature_cols = [col for col in train.columns if col not in excluded_cols]

print("Target:", target_col)
print("Leakage columns:", leakage_cols)
print("Excluded columns:", excluded_cols)
print("\nFeature columns:")
print(feature_cols)
print("\nNumber of features:", len(feature_cols))

Target: count
Leakage columns: ['casual', 'registered']
Excluded columns: ['count', 'casual', 'registered', 'datetime', 'day']

Feature columns:
['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'year', 'month', 'hour', 'weekday', 'is_weekend']

Number of features: 13


In [6]:
X = train[feature_cols].copy()
y = train[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

X shape: (10886, 13)
y shape: (10886,)


,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,year,month,hour,weekday,is_weekend
0,1,0,0,1,9.84,14.395,81,0.0,2011,1,0,5,1
1,1,0,0,1,9.02,13.635,80,0.0,2011,1,1,5,1
2,1,0,0,1,9.02,13.635,80,0.0,2011,1,2,5,1
3,1,0,0,1,9.84,14.395,75,0.0,2011,1,3,5,1
4,1,0,0,1,9.84,14.395,75,0.0,2011,1,4,5,1


0    16
1    40
2    32
3    13
4     1
Name: count, dtype: int64

In [7]:
local_train_mask = train["day"] <= 15
local_valid_mask = train["day"].between(16, 19)

X_train = X.loc[local_train_mask].copy()
y_train = y.loc[local_train_mask].copy()

X_valid = X.loc[local_valid_mask].copy()
y_valid = y.loc[local_valid_mask].copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_valid shape:", y_valid.shape)

X_train shape: (8600, 13)
y_train shape: (8600,)
X_valid shape: (2286, 13)
y_valid shape: (2286,)


In [8]:
print("Local train days:")
print(train.loc[local_train_mask, "day"].value_counts().sort_index())

print("\nLocal validation days:")
print(train.loc[local_valid_mask, "day"].value_counts().sort_index())

print("\nLocal train datetime range:")
print(train.loc[local_train_mask, "datetime"].min())
print(train.loc[local_train_mask, "datetime"].max())

print("\nLocal validation datetime range:")
print(train.loc[local_valid_mask, "datetime"].min())
print(train.loc[local_valid_mask, "datetime"].max())

Local train days:
day
1     575
2     573
3     573
4     574
5     575
6     572
7     574
8     574
9     575
10    572
11    568
12    573
13    574
14    574
15    574
Name: count, dtype: int64

Local validation days:
day
16    574
17    575
18    563
19    574
Name: count, dtype: int64

Local train datetime range:
2011-01-01 00:00:00
2012-12-15 23:00:00

Local validation datetime range:
2011-01-16 00:00:00
2012-12-19 23:00:00


## Stage 5 setup notes

This notebook performs a controlled log-target experiment and limited tuning.

Data used:

- only `train.csv`

Official Kaggle `test.csv` is not used for validation.

Target:

- `count`

Main feature set excludes:

- `count`
- `casual`
- `registered`
- raw `datetime`
- `day`

Leakage control:

- `casual` and `registered` are excluded because they are target components.
- raw `datetime` is excluded after datetime-derived feature extraction.
- `day` is excluded from the main comparison because it may transfer poorly to official test days 20–31.

Local validation split:

- local train: days 1–15
- local validation: days 16–19

This is the same validation scheme used in Stage 3 and Stage 4.

No official test data is used.
No random split is used.
No Kaggle submission is created.
No final model is saved.

In [9]:
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error

In [10]:
def clipped_predictions(y_pred):
    return np.clip(y_pred, a_min=0, a_max=None)


def regression_metrics(y_true, y_pred):
    y_pred_clipped = clipped_predictions(y_pred)

    rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred_clipped))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_clipped))
    mae = mean_absolute_error(y_true, y_pred_clipped)

    return {
        "RMSLE": rmsle,
        "RMSE": rmse,
        "MAE": mae,
        "min_pred_raw": np.min(y_pred),
        "min_pred_clipped": np.min(y_pred_clipped),
        "max_pred_raw": np.max(y_pred),
    }

In [11]:
candidate_models = {
    "RandomForestRegressor": RandomForestRegressor(random_state=42, n_jobs=-1),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=42, n_jobs=-1),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(random_state=42),
}

Raw vs log target evaluation

In [12]:
target_strategy_results = []

for model_name, model in candidate_models.items():
    # Raw target
    model.fit(X_train, y_train)
    y_pred_raw_target = model.predict(X_valid)

    raw_metrics = regression_metrics(y_valid, y_pred_raw_target)
    raw_metrics["model"] = model_name
    raw_metrics["target_strategy"] = "raw_count"
    target_strategy_results.append(raw_metrics)

    # Log1p target
    model_log = candidate_models[model_name].__class__(**model.get_params())
    model_log.fit(X_train, np.log1p(y_train))
    y_pred_log_target = np.expm1(model_log.predict(X_valid))

    log_metrics = regression_metrics(y_valid, y_pred_log_target)
    log_metrics["model"] = model_name
    log_metrics["target_strategy"] = "log1p_count"
    target_strategy_results.append(log_metrics)

target_strategy_df = (
    pd.DataFrame(target_strategy_results)
    .set_index(["target_strategy", "model"])
    .sort_values("RMSLE")
)

target_strategy_df

RMSLE       RMSE        MAE  \
target_strategy model                                                           
log1p_count     HistGradientBoostingRegressor  0.309848  48.788159  29.053384   
                RandomForestRegressor          0.337034  53.100786  31.310561   
                ExtraTreesRegressor            0.339971  53.354813  31.450967   
raw_count       RandomForestRegressor          0.352386  52.494191  30.632303   
                ExtraTreesRegressor            0.360769  51.676713  30.595973   
                HistGradientBoostingRegressor  0.451918  45.782780  27.786296   

                                               min_pred_raw  min_pred_clipped  \
target_strategy model                                                           
log1p_count     HistGradientBoostingRegressor      0.674674          0.674674   
                RandomForestRegressor              1.379466          1.379466   
                ExtraTreesRegressor                1.192606          1.192606   
raw_count       RandomForestRegressor              1.760000          1.760000   
                ExtraTreesRegressor                1.130000          1.130000   
                HistGradientBoostingRegressor    -17.168972          0.000000   

                                               max_pred_raw  
target_strategy model                                        
log1p_count     HistGradientBoostingRegressor    811.159871  
                RandomForestRegressor            925.544552  
                ExtraTreesRegressor              914.490272  
raw_count       RandomForestRegressor            926.280000  
                ExtraTreesRegressor              896.530000  
                HistGradientBoostingRegressor    905.601978

Cleaner comparison view

In [13]:
target_strategy_summary = (
    target_strategy_df
    .reset_index()
    .pivot(index="model", columns="target_strategy", values=["RMSLE", "RMSE", "MAE"])
)

target_strategy_summary

RMSLE                  RMSE             \
target_strategy               log1p_count raw_count log1p_count  raw_count   
model                                                                        
ExtraTreesRegressor              0.339971  0.360769   53.354813  51.676713   
HistGradientBoostingRegressor    0.309848  0.451918   48.788159  45.782780   
RandomForestRegressor            0.337034  0.352386   53.100786  52.494191   

                                      MAE             
target_strategy               log1p_count  raw_count  
model                                                 
ExtraTreesRegressor             31.450967  30.595973  
HistGradientBoostingRegressor   29.053384  27.786296  
RandomForestRegressor           31.310561  30.632303

Compare with Stage 4 references

In [14]:
stage4_rf_rmsle = 0.352386
stage4_hgb_rmse = 45.782780
stage4_hgb_mae = 27.786296

best_untuned_stage5 = target_strategy_df.sort_values("RMSLE").head(1)
best_untuned_stage5

,,RMSLE,RMSE,MAE,min_pred_raw,min_pred_clipped,max_pred_raw
target_strategy,model,,,,,,
log1p_count,HistGradientBoostingRegressor,0.309848,48.788159,29.053384,0.674674,0.674674,811.159871


### Raw vs log1p target findings

- Same feature set was used for all models.
- Same time-aware split was used:
  - local train: days 1–15
  - local validation: days 16–19
- Official Kaggle `test.csv` was not used.
- `casual` and `registered` were not used.
- No hyperparameter tuning was performed in this block.

## Limited tuning with log1p target

In [15]:
from itertools import product

In [16]:
search_spaces = {
    "RandomForestRegressor": [
        {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 1},
        {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 1},
        {"n_estimators": 300, "max_depth": 20, "min_samples_leaf": 1},
        {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 2},
    ],
    "ExtraTreesRegressor": [
        {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 1},
        {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 1},
        {"n_estimators": 300, "max_depth": 20, "min_samples_leaf": 1},
        {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 2},
    ],
    "HistGradientBoostingRegressor": [
        {"max_iter": 100, "learning_rate": 0.1, "max_leaf_nodes": 31, "l2_regularization": 0.0},
        {"max_iter": 200, "learning_rate": 0.05, "max_leaf_nodes": 31, "l2_regularization": 0.0},
        {"max_iter": 300, "learning_rate": 0.03, "max_leaf_nodes": 31, "l2_regularization": 0.0},
        {"max_iter": 200, "learning_rate": 0.05, "max_leaf_nodes": 63, "l2_regularization": 0.0},
        {"max_iter": 200, "learning_rate": 0.05, "max_leaf_nodes": 31, "l2_regularization": 0.1},
    ],
}

Factory function

In [17]:
def make_model(model_name, params):
    if model_name == "RandomForestRegressor":
        return RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
            **params,
        )

    if model_name == "ExtraTreesRegressor":
        return ExtraTreesRegressor(
            random_state=42,
            n_jobs=-1,
            **params,
        )

    if model_name == "HistGradientBoostingRegressor":
        return HistGradientBoostingRegressor(
            random_state=42,
            **params,
        )

    raise ValueError(f"Unknown model: {model_name}")

Manual limited tuning on fixed local validation

In [18]:
tuning_results = []

for model_name, param_grid in search_spaces.items():
    for params in param_grid:
        model = make_model(model_name, params)

        model.fit(X_train, np.log1p(y_train))
        y_pred = np.expm1(model.predict(X_valid))

        metrics = regression_metrics(y_valid, y_pred)
        metrics["model"] = model_name
        metrics["target_strategy"] = "log1p_count"
        metrics["params"] = params
        tuning_results.append(metrics)

tuning_results_df = (
    pd.DataFrame(tuning_results)
    .sort_values("RMSLE")
    .reset_index(drop=True)
)

tuning_results_df

,RMSLE,RMSE,MAE,min_pred_raw,min_pred_clipped,max_pred_raw,model,target_strategy,params
0,0.306045,47.065693,28.274786,1.138463,1.138463,825.950430,HistGradientBoostingRegressor,log1p_count,"{'max_iter': 200, 'learning_rate': 0.05, 'max_..."
1,0.307263,47.074728,28.287317,0.952891,0.952891,852.033112,HistGradientBoostingRegressor,log1p_count,"{'max_iter': 200, 'learning_rate': 0.05, 'max_..."
2,0.307452,47.291682,28.632527,0.923111,0.923111,830.148754,HistGradientBoostingRegressor,log1p_count,"{'max_iter': 300, 'learning_rate': 0.03, 'max_..."
3,0.309848,48.788159,29.053384,0.674674,0.674674,811.159871,HistGradientBoostingRegressor,log1p_count,"{'max_iter': 100, 'learning_rate': 0.1, 'max_l..."
4,0.312650,47.349328,27.815020,1.226912,1.226912,868.840848,HistGradientBoostingRegressor,log1p_count,"{'max_iter': 200, 'learning_rate': 0.05, 'max_..."
5,0.332838,52.037375,30.518176,1.210181,1.210181,901.357651,ExtraTreesRegressor,log1p_count,"{'n_estimators': 300, 'max_depth': None, 'min_..."
6,0.334046,52.838248,30.995353,1.459101,1.459101,914.688620,RandomForestRegressor,log1p_count,"{'n_estimators': 200, 'max_depth': None, 'min_..."
7,0.334194,52.682021,30.853793,1.464884,1.464884,907.397850,RandomForestRegressor,log1p_count,"{'n_estimators': 300, 'max_depth': None, 'min_..."
8,0.334945,52.979770,31.063814,1.460564,1.460564,913.899793,RandomForestRegressor,log1p_count,"{'n_estimators': 300, 'max_depth': None, 'min_..."
9,0.335455,52.899604,30.976939,1.461958,1.461958,912.781793,RandomForestRegressor,log1p_count,"{'n_estimators': 300, 'max_depth': 20, 'min_sa..."


Best params by model

In [19]:
best_tuned_by_model = (
    tuning_results_df
    .sort_values("RMSLE")
    .groupby("model", as_index=False)
    .first()
    .sort_values("RMSLE")
)

best_tuned_by_model

,model,RMSLE,RMSE,MAE,min_pred_raw,min_pred_clipped,max_pred_raw,target_strategy,params
1,HistGradientBoostingRegressor,0.306045,47.065693,28.274786,1.138463,1.138463,825.950430,log1p_count,"{'max_iter': 200, 'learning_rate': 0.05, 'max_..."
0,ExtraTreesRegressor,0.332838,52.037375,30.518176,1.210181,1.210181,901.357651,log1p_count,"{'n_estimators': 300, 'max_depth': None, 'min_..."
2,RandomForestRegressor,0.334046,52.838248,30.995353,1.459101,1.459101,914.688620,log1p_count,"{'n_estimators': 200, 'max_depth': None, 'min_..."


Compare untuned vs tuned

In [20]:
untuned_log_results = (
    target_strategy_df
    .reset_index()
    .query("target_strategy == 'log1p_count'")
    .copy()
)

untuned_log_results["params"] = "default"
untuned_log_results["run_type"] = "untuned_log1p"

best_tuned_by_model_compare = best_tuned_by_model.copy()
best_tuned_by_model_compare["run_type"] = "limited_tuned_log1p"

tuned_vs_untuned_df = (
    pd.concat([
        untuned_log_results[
            ["run_type", "model", "RMSLE", "RMSE", "MAE", "min_pred_raw", "max_pred_raw", "params"]
        ],
        best_tuned_by_model_compare[
            ["run_type", "model", "RMSLE", "RMSE", "MAE", "min_pred_raw", "max_pred_raw", "params"]
        ],
    ])
    .sort_values("RMSLE")
    .reset_index(drop=True)
)

tuned_vs_untuned_df

,run_type,model,RMSLE,RMSE,MAE,min_pred_raw,max_pred_raw,params
0,limited_tuned_log1p,HistGradientBoostingRegressor,0.306045,47.065693,28.274786,1.138463,825.950430,"{'max_iter': 200, 'learning_rate': 0.05, 'max_..."
1,untuned_log1p,HistGradientBoostingRegressor,0.309848,48.788159,29.053384,0.674674,811.159871,default
2,limited_tuned_log1p,ExtraTreesRegressor,0.332838,52.037375,30.518176,1.210181,901.357651,"{'n_estimators': 300, 'max_depth': None, 'min_..."
3,limited_tuned_log1p,RandomForestRegressor,0.334046,52.838248,30.995353,1.459101,914.688620,"{'n_estimators': 200, 'max_depth': None, 'min_..."
4,untuned_log1p,RandomForestRegressor,0.337034,53.100786,31.310561,1.379466,925.544552,default
5,untuned_log1p,ExtraTreesRegressor,0.339971,53.354813,31.450967,1.192606,914.490272,default


Compare with Stage 4 best

In [21]:
stage4_best = {
    "model": "RandomForestRegressor",
    "target_strategy": "raw_count",
    "RMSLE": 0.352386,
    "RMSE": 52.494191,
    "MAE": 30.632303,
    "note": "Stage 4 best by RMSLE",
}

stage5_best = tuning_results_df.iloc[0].to_dict()

print("Stage 4 best RMSLE:", stage4_best["RMSLE"])
print("Stage 5 best tuned RMSLE:", stage5_best["RMSLE"])
print("RMSLE delta:", stage5_best["RMSLE"] - stage4_best["RMSLE"])

stage5_best

Stage 4 best RMSLE: 0.352386
Stage 5 best tuned RMSLE: 0.3060449555471268
RMSLE delta: -0.04634104445287318


{'RMSLE': 0.3060449555471268,
 'RMSE': 47.06569264085444,
 'MAE': 28.274785977672185,
 'min_pred_raw': 1.1384632117959232,
 'min_pred_clipped': 1.1384632117959232,
 'max_pred_raw': 825.950429634485,
 'model': 'HistGradientBoostingRegressor',
 'target_strategy': 'log1p_count',
 'params': {'max_iter': 200,
  'learning_rate': 0.05,
  'max_leaf_nodes': 31,
  'l2_regularization': 0.1}}

### Limited tuning findings

- Tuning was limited to small predefined search spaces.
- No `GridSearchCV` or `RandomizedSearchCV` was used.
- The same time-aware validation split was used.
- All tuned models used `log1p(count)` target and inverse `expm1` prediction transformation.
- Official Kaggle `test.csv` was not used.


## Stage 5 conclusions

### Controlled setup

This notebook used the same controlled setup as Stage 4:

- data source: `train.csv` only
- target: `count`
- local train: days 1–15
- local validation: days 16–19
- official Kaggle `test.csv` was not used
- random split was not used
- `day` was excluded from the main feature set
- `casual` and `registered` were excluded as target components
- raw `datetime` was excluded after feature extraction

### Main feature set

- `season`
- `holiday`
- `workingday`
- `weather`
- `temp`
- `atemp`
- `humidity`
- `windspeed`
- `year`
- `month`
- `hour`
- `weekday`
- `is_weekend`

Excluded from `X`:

- `count`
- `casual`
- `registered`
- raw `datetime`
- `day`

### Target strategies compared

Two target strategies were compared:

1. Raw target:
   - fit on `count`
   - predict `count`

2. Log target:
   - fit on `log1p(count)`
   - predict in log space
   - inverse transform with `expm1`

Metrics were computed on the original `count` scale.

### Untuned raw vs log1p target comparison

Best untuned result by RMSLE:

- `HistGradientBoostingRegressor` with `log1p(count)`

Metrics:

- RMSLE: 0.309848
- RMSE: 48.788159
- MAE: 29.053384

The `log1p` target improved RMSLE for all three candidate models:

- `RandomForestRegressor`: 0.352386 → 0.337034
- `ExtraTreesRegressor`: 0.360769 → 0.339971
- `HistGradientBoostingRegressor`: 0.451918 → 0.309848

### Limited tuning

Limited tuning was performed only for promising candidates:

- `RandomForestRegressor`
- `ExtraTreesRegressor`
- `HistGradientBoostingRegressor`

No large `GridSearchCV` or `RandomizedSearchCV` was used.

All tuned models used:

- `log1p(count)` target
- inverse prediction transform with `expm1`
- the same fixed local validation split

### Best tuned model

Best tuned model by primary metric RMSLE:

- `HistGradientBoostingRegressor`

Best params:

```text
{
    "max_iter": 200,
    "learning_rate": 0.05,
    "max_leaf_nodes": 31,
    "l2_regularization": 0.1
}

Validation metrics:

RMSLE: 0.306045
RMSE: 47.065693
MAE: 28.274786
Comparison with Stage 4

Stage 4 best model by RMSLE:

RandomForestRegressor with raw target
RMSLE: 0.352386
RMSE: 52.494191
MAE: 30.632303

Stage 5 best model:

tuned HistGradientBoostingRegressor with log1p(count)
RMSLE: 0.306045
RMSE: 47.065693
MAE: 28.274786

RMSLE improvement:

0.352386 → 0.306045
delta: -0.046341
Main finding

The main improvement came from the log1p(count) target transformation.

Limited tuning gave a smaller additional improvement:

untuned HGB log1p RMSLE: 0.309848
tuned HGB log1p RMSLE: 0.306045
Candidate for next stage

Recommended candidate for the next approved stage:

HistGradientBoostingRegressor
target strategy: log1p(count)
params:
max_iter=200
learning_rate=0.05
max_leaf_nodes=31
l2_regularization=0.1
Leakage and validation checks

Passed:

official Kaggle test.csv was not used
random split was not used
casual not in X
registered not in X
count not in X
raw datetime not in X
day excluded from main comparison
no Kaggle submission created
no final model saved
Important caution

The tuned result is selected using one fixed local validation split.

This can still overfit to the local validation days 16–19, so the result should not be treated as final generalization performance.


---

## 4. Что должно получиться

Notebook должен быть сохранён:

```text
notebooks/04_log_target_and_tuning.ipynb